In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

import os
os.chdir('/kaggle/working/')

## Crawl chứng khoán từ VnExpress


In [ ]:
!pip install newspaper3k -q

In [ ]:
!curl https://raw.githubusercontent.com/codelucas/newspaper/master/download_corpora.py | python -q

In [ ]:
!pip install lxml_html_clean -q

In [ ]:
from newspaper import Article
from bs4 import BeautifulSoup
import urllib.request
import requests

from re import sub
import re
import time
from urllib.error import URLError, HTTPError
import pandas as pd

In [ ]:
def fetch_page(url, retries=3, delay=3):
    for i in range(retries):
        try:
            page = urllib.request.urlopen(url, timeout=10)
            return page
        except urllib.error.URLError as e:
            print(f"Attempt {i+1} failed: {e}")
            time.sleep(delay)
    return None

In [ ]:
def get_links_in_page_vnexpress(url):
    # page = urllib.request.urlopen(url)
    page = fetch_page(url)
    if page is None:
      print(f"Failed to fetch page {url}")
      return None
    soup = BeautifulSoup(page, 'html.parser')
    # <section>
    sections = soup.find_all('section', attrs={'class':'section section_container mt15'})

    # <h2>
    h_all = []
    for section in sections:
        h_all.extend(section.find_all('h2', attrs={'class':'title-news'}))

    # <a>
    a_all = []
    results = [[]]
    for h in h_all:
        a_all.extend(h.find_all('a', attrs={'class':''}))
    for a in a_all:
        title = a.get('title')
        link = a.get('href')
        print('Title: {} - Link: {}'.format(title, link))
        if title == None or link == None:
          continue
        results.append([title,link])
    return results

In [ ]:
get_links_in_page_vnexpress('https://vnexpress.net/kinh-doanh/chung-khoan-p2')

In [ ]:
def get_links():
    news_list = []
    urls = [f'https://vnexpress.net/kinh-doanh/chung-khoan-p{i}' for i in range(1, 21)]
    for url in urls:
        results = get_links_in_page_vnexpress(url)
        if results:
            for result in results:
                if len(result) >= 2:
                    news = {
                        'title': result[0],
                        'link': result[1]
                    }
                    news_list.append(news)

    df = pd.DataFrame(news_list)
    return df

df = get_links()
df.drop_duplicates(subset=['link'], inplace=True)
df.reset_index(drop=True, inplace=True)
print(df.shape[0])
# df.to_csv('data/news-link.csv', index=False)

## Cần lọc ra những bài báo mới so với database
- Chưa giải quyết được

In [ ]:
def crawl_by_url(url, retries=3, backoff_factor=0.3):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/85.0.4183.121 Safari/537.36'
    }

    for i in range(retries):
        try:
            response = requests.get(url, headers=headers, allow_redirects=False)
            if response.status_code == 200:
                # tìm thẻ có class = date
                soup = BeautifulSoup(response.text, 'html.parser')
                
                if soup.find('span', class_='date'):
                    date = soup.find('span', class_='date').text
                else :
                    date = None

                article = Article(url, language='vi')
                article.set_html(response.text)
                article.download(input_html=response.text)
                article.parse()

                contents = str(article.text).strip()
                article.nlp()

                return date, contents
            else:
                print(f"Failed to crawl {url}, status code: {response.status_code}")
                return None, None
        except requests.exceptions.RequestException as e:
            print(f"Error occurred during crawling: {e}")
            time.sleep(backoff_factor * (2 ** i))

    return None, None

# create content collumns
# df = pd.read_csv('/kaggle/input/vnexpress-links/news-link.csv')
df['content'] = ''
df['date'] = ''
n = df.shape[0]

for index, row in df.iterrows():

    print(f"Processing {index+1}/{n} ")
    result = crawl_by_url(row['link'])
    df.at[index, 'date'] = result[0]
    df.at[index, 'content'] = result[1]


print(n)
df = df[df['content'].notnull()]
# remove duplicate
print(df['content'].duplicated().sum())
df = df.drop_duplicates(subset='content')

print(f"Number of rows after removing duplicates: {df.shape[0]}")
df.to_csv('news_cleaned.csv', index=False)